In [ ]:
import os

import tinker

from tinker_cookbook import renderers
from tinker_cookbook.supervised.data import conversation_to_datum
from tinker_cookbook.tokenizer_utils import get_tokenizer

BASE_MODEL = "Qwen/Qwen3.5-4B"

service_client = tinker.ServiceClient()
training_client = await service_client.create_lora_training_client_async(
    base_model=BASE_MODEL, rank=16
)

_tokenizer = get_tokenizer(BASE_MODEL)
_renderer = renderers.get_renderer("qwen3_5_disable_thinking", _tokenizer)
_messages = [
    {"role": "user", "content": "What is Tinker?"},
    {
        "role": "assistant",
        "content": "Tinker is a cloud training API for LLM fine-tuning.",
    },
]
_datum = conversation_to_datum(_messages, _renderer, max_length=512)

_fwd = await training_client.forward_backward_async([_datum], loss_fn="cross_entropy")
_opt = await training_client.optim_step_async(tinker.AdamParams(learning_rate=1e-4))
await _fwd.result_async()
await _opt.result_async()

_save_result = training_client.save_weights_for_sampler(name="adapter-tutorial")
sampler_path = _save_result.result().path
print(f"Base model:  {BASE_MODEL}")
print(f"Checkpoint:  {sampler_path}")